# Setup

In [ ]:
"""Baseline RAG: ask real analyst questions, get cited answers."""
import logging

from finintel.agent.generator import RAGPipeline

logging.basicConfig(level=logging.WARNING)  # quiet by default for cleaner notebook output
%load_ext autoreload
%autoreload 2

# Loading the embedder takes 5-10s; do it once
rag = RAGPipeline()
print(f"Model:  {rag.model}")
print(f"Top-K:  {rag.top_k}")
print(f"Vector dim: {rag.embedder.dim}")

# A focused single-company question

In [ ]:
result = rag.answer(
    "How does Google describe risks from AI in its most recent 10-K? Be specific about which categories of risk they identify.",
    ticker="GOOGL",
    section="risk_factors",
)
print("=" * 70)
print(f"Q: {result.question}\n")
print(result.answer)
print("\n" + "=" * 70)
print(f"Sources used ({len(result.sources)}):")
for s in result.sources:
    print(f"  [{s['chunk_id']}] score={s['score']:.3f}")
print(f"\nTokens: {result.input_tokens:,} in / {result.output_tokens:,} out")
# print(f"Approx cost: ${(result.input_tokens * 3 + result.output_tokens * 15) / 1_000_000:.4f}")
print(f"On Groq free tier — no charge.")

# A multi-company comparison (the kind of question that breaks naive RAG)

In [ ]:
result = rag.answer(
    "Compare how AAPL, MSFT, and GOOGL discuss cybersecurity risks. "
    "Which one frames it most aggressively?"
)
print(result.answer)
print("\nSources:")
for s in result.sources:
    print(f"  [{s['chunk_id']}] {s['ticker']} score={s['score']:.3f}")

# A question that REQUIRES MD&A, not Risk Factors

In [ ]:
result = rag.answer(
    "What was the impact of foreign exchange rates on Alphabet's revenue? "
    "Include the constant-currency adjustment they discuss.",
    ticker="GOOGL",
    section="mda",
)
print(result.answer)

# A question the corpus genuinely can't answer (test refusal)

In [ ]:
result = rag.answer(
    "What is the current stock price of Apple?"  # not in any 10-K
)
print(result.answer)
# Expected: explicit refusal, possibly suggesting a refined query